In [1]:
import os
import sys
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
os.chdir('..') 
    
from pathlib import Path
from chromadb import PersistentClient

from app.config import get_settings
from ingestion.loaders import load_pdf_pages, chunk_pages
from ingestion.build_index import embedd_chunks
from app.rag_pipeline import retrieve

In [2]:
settings = get_settings()
client = PersistentClient(path=settings.index_dir)
collection = client.get_collection(settings.collection_name)

In [3]:
get_settings().openai_api_key[:8] + "..."

'sk-proj-...'

In [4]:
DATA_PATH = Path("./data/NVIDIAAn.pdf")
DATA_PATH

PosixPath('data/NVIDIAAn.pdf')

- #### Check loading

In [5]:
%%time
pages = list(load_pdf_pages(path=DATA_PATH))
first_page = pages[0]
print(f"first page nr:{first_page.page_number}.")
print(f"Initial text:\n{first_page.text[:200]}.")

first page nr:1.
Initial text:
NVIDIA Announces Financial Results for Second Quarter
Fiscal 2024
Record revenue of $13.51 billion, up 88% from Q1, up 101% from year ago
Record Data Center revenue of $10.32 billion, up 141% from Q1,.
CPU times: user 1.88 s, sys: 9.01 ms, total: 1.89 s
Wall time: 1.9 s


- #### Check loading

In [6]:
%%time
chunks = list(chunk_pages(pages))
assert 500 < chunks[0].length <= 800, f"Bad length: {chunks[0].length}"

CPU times: user 122 μs, sys: 26 μs, total: 148 μs
Wall time: 154 μs


In [7]:
print(f"{len(pages)} pages and {len(chunks)} chunks")
print(f"sampel ID {chunks[-1].id}")
print(f"avg chunk len = {int(sum(c.length for c in chunks) / len(chunks))} chars")
print(f"Example chunk: {chunks[-1].text[:50]}...")
# assert all(c.page in range(1, len(pages)+1) for c in chunks)

9 pages and 45 chunks
sampel ID p9_c7
avg chunk len = 709 chars
Example chunk:  other
countries. Other company and product names ...


In [8]:
DATA_PATH.stem

'NVIDIAAn'

- #### Check embedding vectorization

In [9]:
%%time
embeddings = embedd_chunks(chunks[:2])

CPU times: user 306 ms, sys: 14.4 ms, total: 320 ms
Wall time: 2.06 s


In [10]:
print(f"There are {len(embeddings)} embeddings")
print(
    f"Size of embedding for {get_settings().openai_embedding_model} model is {len(embeddings[0])} dimensions"
)  # Should be 1536
print(f"Embedding values range from {min(embeddings[0]):.3f} to {max(embeddings[0]):.3f}")
print(f"Example embedding: {[round(emb, 3) for emb in embeddings[0][:5]]}...")

There are 2 embeddings
Size of embedding for text-embedding-3-small model is 1536 dimensions
Embedding values range from -0.087 to 0.083
Example embedding: [0.028, -0.03, 0.037, 0.004, 0.034]...


- #### Check question related chunk search

In [11]:
%%time
question = "What was NVIDIA's total revenue for Q2 FY2024??"
top_k = 3

chunks_search = retrieve(question, top_k=top_k)
print(f"Found {len(chunks_search)} chunks:")
for i, chunk in enumerate(chunks_search, 1):
    print(f"\n{i}. Score: {chunk.score:.3f} in Page: {chunk.page}")
    print(f"   ID: {chunk.id}")
    print(f"   Text: {chunk.text[:50]}...")

Found 3 chunks:

1. Score: 0.776 in Page: 1
   ID: p1_c1
   Text: NVIDIA Announces Financial Results for Second Quar...

2. Score: 0.771 in Page: 2
   ID: p2_c1
   Text: NVIDIA’s outlook for the third quarter of fiscal 2...

3. Score: 0.750 in Page: 1
   ID: p1_c3
   Text: .38 billion to shareholders in the form of 7.5 mil...
CPU times: user 42.8 ms, sys: 13.9 ms, total: 56.8 ms
Wall time: 919 ms
